# SAM ViT-B — DIMER promptable image segmentation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/tutorials/sam_vit_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fsam--vit--base-ffcc4d?style=flat)](https://huggingface.co/facebook/sam-vit-base) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fsegment--anything-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/segment-anything) [![arXiv](https://img.shields.io/badge/arXiv-2304.02643-b31b1b.svg)](https://arxiv.org/abs/2304.02643)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** promptable image segmentation (point and/or box prompts → masks for one object) using the pinned `facebook/sam-vit-base` weights (SAM v1, ViT-B)

**This notebook is standalone.** It carries the repository's pipeline module (`src/sam_vit_segmentation_pipeline/pipeline.py` at revision `70c2f494743b`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `70c1a07f894ebb5b307fd9eaaee97b9dfc16068f` (~375 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

At inference the image's longest edge is resized to 1024 px and padded to 1024×1024, then encoded once by a ViT-B image encoder; the prompt encoder embeds your clicks (label 1 = foreground, 0 = background) and/or one xyxy box, and the mask decoder returns up to three candidate masks at 256×256, which the pipeline up-samples to the input resolution (stripping the padding) and binarises at logit 0, together with the model's own predicted IoU for each candidate. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and processor, and the carried pipeline module adds snapshot verification, image and prompt validation with named ceilings, a single-object-per-call contract, a fixed output contract and the `mask_iou`, `validate_inputs` and `evaluation_report` helpers. This is the original 2023 SAM; the sibling `sam2-segmentation-pipeline` packages SAM 2.1 with the same contract, and the two are not compared here. The default sample is a synthetic scene drawn in code; the IoU reported for it is sanity evidence against a shape you drew, not a benchmark claim.

**No load notice is expected.** The card-pass smoke wrote nothing to stderr during the load, so Section 3 should print only the dictionaries it is asked for. A warning or an error there is a real signal.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic scene with a known object, validate the image and prompts into an input manifest through the pipeline's own validation stage, segment one object from a click through the public API, read the candidate masks and their uncalibrated model-predicted IoU scores correctly (including why a score can exceed 1.0), produce an evaluation report that is `sample-sanity` with `mask_iou` only when a reference mask exists and `not-measurable` otherwise, exercise an optional BYOD path, and export the mask plus machine-readable provenance.

**This notebook does not demonstrate:** automatic "segment everything" mask generation (the upstream grid-prompt pipeline is not wrapped), text prompts (see the sibling Grounding DINO pipeline for boxes from text), mask-input prompts, several objects in one call, semantic class labels, video, mIoU evaluation against labelled masks, or any training. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 3.77 s to load (after a 0.21 s manifest verification) and 2.93 s for the click prompt on the 320×240 synthetic scene in the Windows venv (Intel Core Ultra 9 275HX); the encoder cost is fixed by the 1024×1024 working size, so image resolution only changes the size of the returned masks (a 4096×4096 box prompt took 3.11 s in the same smoke). The pinned `torch==2.14.0` install and the 375 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what a binary mask is; what intersection-over-union measures.
- **Expected output:** no loader notice is expected. The card-pass smoke wrote nothing to stderr during the load, so Section 3 should print only the dictionaries it is asked for; a warning or an error there is a real signal. The upstream repository's `pytorch_model.bin` and TensorFlow weights are not in the manifest and are never staged or loaded.
- **Data:** the default sample is a deterministic 320×240 scene drawn in code (grey background, one dark rectangle, one red disc) with a foreground click inside the rectangle, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus a click position inside it set through the form parameters. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/sam-vit-base` snapshot (~375 MB) at revision `70c1a07f894e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'sam-vit-segmentation-pipeline',
    'repository_revision': '70c2f494743b6fb0c3abfe15f2082fb94bf39ec1',
    'embedded_module': 'src/sam_vit_segmentation_pipeline/pipeline.py',
    'module_sha256': 'a83f3bc52c597ab89bcc786d56e93b322c669f38237742679e3509400566d42c',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/sam_vit_segmentation_pipeline/pipeline.py` @ `70c2f494743b`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
"""Promptable image segmentation over the pinned ``facebook/sam-vit-base`` (SAM v1, ViT-B) checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``segment``: one object per call from point
clicks and/or one box, returning boolean masks at input resolution plus the model's predicted IoU per mask.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "facebook/sam-vit-base"
MODEL_REVISION = "70c1a07f894ebb5b307fd9eaaee97b9dfc16068f"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "sam-vit-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Input ceilings. The processor resizes the longest edge to 1024 and pads to 1024x1024 (see
# preprocessor_config.json), so model cost is fixed; the caller's resolution only sets the size of the
# up-sampled output masks. Prompts are one object per call: up to MAX_PROMPTS point clicks (label 1 =
# foreground, 0 = background) and/or one xyxy box.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
NUM_MULTIMASK_OUTPUTS = 3  # config.json mask_decoder_config.num_multimask_outputs
MASK_THRESHOLD = 0.0  # logits above this become True in the binarised masks (processor default)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-union of two boolean masks of identical shape; the primitive behind any mIoU."""
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape != b.shape:
        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")
    if a.dtype != np.bool_ or b.dtype != np.bool_:
        raise TypeError("mask_iou expects boolean arrays")
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def validate_prompts(
    width: int,
    height: int,
    points: Sequence[Sequence[float]] | None,
    point_labels: Sequence[int] | None,
    box: Sequence[float] | None,
) -> tuple[list[list[float]] | None, list[int] | None, list[float] | None]:
    """Check one object's prompts: points inside the image with 0/1 labels, and/or one xyxy box inside it."""
    if points is None and box is None:
        raise ValueError("provide at least one of points or box")
    clean_points = clean_labels = None
    if points is not None:
        if isinstance(points, str) or not isinstance(points, Sequence):
            raise TypeError("points must be a sequence of [x, y] pairs")
        if not 1 <= len(points) <= MAX_PROMPTS:
            raise ValueError(f"point count {len(points)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
        if point_labels is None or len(point_labels) != len(points):
            raise ValueError("point_labels must be given with one 0/1 entry per point")
        clean_points, clean_labels = [], []
        for (x, y), label in zip(points, point_labels, strict=True):
            if not (0 <= x < width and 0 <= y < height):
                raise ValueError(f"point ({x}, {y}) outside image {width}x{height}")
            if label not in (0, 1) or isinstance(label, bool):
                raise ValueError(f"point label must be 0 or 1, got {label!r}")
            clean_points.append([float(x), float(y)])
            clean_labels.append(int(label))
    elif point_labels is not None:
        raise ValueError("point_labels given without points")
    clean_box = None
    if box is not None:
        if len(box) != 4:
            raise ValueError("box must be [x0, y0, x1, y1]")
        x0, y0, x1, y1 = (float(v) for v in box)
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError(f"box {box!r} is not a non-empty xyxy box inside image {width}x{height}")
        clean_box = [x0, y0, x1, y1]
    return clean_points, clean_labels, clean_box


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one PIL.Image.Image (any mode, converted to RGB) plus one object's prompts: point clicks "
        "and/or one xyxy box"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "points": [1, MAX_PROMPTS],
    "point_labels": "one per point, 1 = foreground and 0 = background",
    "box": "at most one [x0, y0, x1, y1] inside the image with x0 < x1 and y0 < y1",
    "objects_per_call": 1,
    "multimask_outputs": NUM_MULTIMASK_OUTPUTS,
    "preprocessing": (
        "image converted to RGB; the processor resizes the longest edge to 1024 and pads to 1024x1024; "
        "returned masks are up-sampled "
        f"to the input resolution and binarised at logit MASK_THRESHOLD={MASK_THRESHOLD}"
    ),
}


def _check_inputs(
    image: Any,
    points: Any,
    point_labels: Any,
    box: Any,
    multimask: Any,
) -> tuple[Image.Image, list[list[float]] | None, list[int] | None, list[float] | None]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the cleaned request.

    ``segment`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    clean_points, clean_labels, clean_box = validate_prompts(
        rgb.width, rgb.height, points, point_labels, box
    )
    if not isinstance(multimask, bool):
        raise TypeError("multimask must be a bool")
    return rgb, clean_points, clean_labels, clean_box


def validate_inputs(
    image: Image.Image,
    *,
    points: Sequence[Sequence[float]] | None = None,
    point_labels: Sequence[int] | None = None,
    box: Sequence[float] | None = None,
    multimask: bool = True,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, clean_points, clean_labels, clean_box = _check_inputs(
        image, points, point_labels, box, multimask
    )
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (segment takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_points": 0 if clean_points is None else len(clean_points),
                "has_box": clean_box is not None,
            }
        ],
        "points": clean_points,
        "point_labels": clean_labels,
        "box": clean_box,
        "multimask": multimask,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_mask: Any = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With a boolean ``reference_mask`` of the same shape as the returned masks the report carries one
    ``mask_iou`` entry per candidate as sample-sanity geometry evidence; without one the verdict is
    ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    masks = np.asarray(result["masks"])
    scores = [float(v) for v in result["iou_scores"]]
    best = int(np.argmax(scores)) if scores else None
    base = {
        "task": "promptable single-object image segmentation (point and/or box prompts)",
        "decision_rule": (
            "keep the candidate with the highest model-predicted IoU; the pipeline ships no "
            "acceptance threshold and does not choose for the caller"
        ),
        "score_semantics": (
            "iou_scores are the model's own uncalibrated predicted IoU for each candidate, not a "
            "measured overlap and not a probability; the regression head is unclipped, so a value "
            "may exceed 1.0"
        ),
        "sample_kind": sample_kind,
        "n_masks": int(masks.shape[0]) if masks.ndim == 3 else 0,
        "best_candidate": best,
        "iou_scores_model_predicted": scores,
        "mask_areas_px": [int(mask.sum()) for mask in masks] if masks.ndim == 3 else [],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference_mask is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth mask was supplied for the evaluated image",
            "needs": (
                "hand-labelled boolean masks for the prompted objects on your own images, scored with "
                "mask_iou per object and averaged into a mean IoU over a held-out set; no labelled "
                "mask set ships with this repository"
            ),
        }
    reference = np.asarray(reference_mask)
    return {
        **base,
        "metrics": [
            {
                "id": "mask_iou",
                "candidate": index,
                "value": mask_iou(masks[index], reference),
                "selected": index == best,
                "estimation": "one reference mask on a single scene, no dispersion estimate",
            }
            for index in range(masks.shape[0])
        ],
        "reference_area_px": int(reference.sum()),
        "verdict": "sample-sanity",
        "reason": (
            "one reference mask on one tutorial sample; geometry sanity evidence, not a segmentation "
            "benchmark"
        ),
        "needs": (
            "a labelled mask set from the deployment domain for any mean-IoU or boundary-quality claim"
        ),
    }


@dataclass
class SAMViTSegmentationPipeline:
    """Promptable image segmentation (points/box -> masks) over the pinned SAM ViT-B checkpoint.

    Prompted mode only (`SamModel` + `SamProcessor`); automatic grid-prompt mask generation is not exposed."""

    _runner: Callable[..., tuple[np.ndarray, list[float]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SAMViTSegmentationPipeline:
        import torch
        from transformers import SamModel, SamProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        processor = SamProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = SamModel.from_pretrained(
            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image, points, labels, box, multimask) -> tuple[np.ndarray, list[float]]:
            prompt_kwargs: dict[str, Any] = {}
            if points is not None:
                prompt_kwargs["input_points"] = [[points]]
                prompt_kwargs["input_labels"] = [[labels]]
            if box is not None:
                prompt_kwargs["input_boxes"] = [[box]]
            inputs = processor(images=image, return_tensors="pt", **prompt_kwargs).to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs, multimask_output=multimask)
            # SAM v1 pads the resized image to 1024x1024; reshaped_input_sizes lets post-processing strip it.
            masks = processor.post_process_masks(
                outputs.pred_masks.cpu(),
                inputs["original_sizes"].cpu(),
                inputs["reshaped_input_sizes"].cpu(),
                mask_threshold=MASK_THRESHOLD,
            )[0]
            return masks[0].numpy().astype(np.bool_), [float(v) for v in outputs.iou_scores[0, 0].tolist()]

        return cls(runner, resolved_device)

    def segment(
        self,
        image: Image.Image,
        *,
        points: Sequence[Sequence[float]] | None = None,
        point_labels: Sequence[int] | None = None,
        box: Sequence[float] | None = None,
        multimask: bool = True,
    ) -> dict[str, Any]:
        """Segment one object; returns K boolean masks (K = 3 with multimask, else 1) at input resolution."""
        rgb, clean_points, clean_labels, clean_box = _check_inputs(
            image, points, point_labels, box, multimask
        )
        masks, iou_scores = self._runner(rgb, clean_points, clean_labels, clean_box, multimask)
        masks = np.asarray(masks)
        expected = (NUM_MULTIMASK_OUTPUTS if multimask else 1, rgb.height, rgb.width)
        if masks.dtype != np.bool_ or masks.shape != expected or len(iou_scores) != expected[0]:
            raise RuntimeError(f"backend returned {masks.shape} {masks.dtype}, {len(iou_scores)} scores")
        return {
            "masks": masks,
            "iou_scores": [float(v) for v in iou_scores],
            "multimask": multimask,
            "points": clean_points,
            "point_labels": clean_labels,
            "box": clean_box,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `70c1a07f894e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `SAMViTSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "sam-vit-base",
  "modelId": "facebook/sam-vit-base",
  "revision": "70c1a07f894ebb5b307fd9eaaee97b9dfc16068f",
  "files": [
    {
      "path": "README.md",
      "bytes": 6727,
      "sha256": "7a8579105233ff1e8fdc6c8387e102e88458e94c873463f640c416de01c921dd"
    },
    {
      "path": "config.json",
      "bytes": 6566,
      "sha256": "5ebd0d8643b486f3a716bf17c2a15531eb818b2b96ff0c6c5dcc88fa015161af"
    },
    {
      "path": "model.safetensors",
      "bytes": 374979480,
      "sha256": "892c410e496344e527255ccdcb2cb7244a609acb5389c7c4fdba1288f861c579"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 466,
      "sha256": "225545a743c654e3c495ec6f545a0eaba57c8ba3fbbd8483b3cb1c0fc58db517"
    }
  ],
  "totalBytes": 374993239
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = SAMViTSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own reference: a deterministic 320×240 RGB scene is drawn in code — grey background, a dark filled rectangle at `[40, 60, 140, 180]` and a red filled disc at `[200, 80, 280, 160]` — and the prompt is one **foreground click at (90, 120)**, inside the rectangle. A boolean reference mask of the drawn rectangle is kept for the `mask_iou` sanity check later. This is the same scene and click the repository's card-pass smoke used; it is not a labelled dataset, so nothing here is an mIoU measurement. The image digest and the prompt are printed. BYOD is optional and disabled by default; when enabled, upload one image and set `POINT_X`/`POINT_Y` to a pixel inside the object you want — no reference mask exists for it, so the evaluation report will be `not-measurable`.

Prompts describe **one object per call**: up to `MAX_PROMPTS` (16) clicks with 0/1 labels and/or one xyxy box; several objects need several calls. `MULTIMASK` (default `True`) asks for three candidate masks — useful when a single click is ambiguous (part, object, or object plus surroundings) — while `False` returns one. Nothing is validated in this cell: the next section hands the image and the prompts to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the click, the multimask setting, and the reference mask area.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
POINT_X = 90  # @param {type:"integer"}
POINT_Y = 120  # @param {type:"integer"}
MULTIMASK = True  # @param {type:"boolean"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    reference_mask = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable.
    image = Image.new('RGB', (320, 240), (128, 128, 128))
    draw = ImageDraw.Draw(image)
    rectangle_box = [40, 60, 140, 180]
    draw.rectangle(rectangle_box, fill=(30, 30, 30))
    draw.ellipse([200, 80, 280, 160], fill=(220, 30, 30))
    reference = Image.new('1', image.size, 0)
    ImageDraw.Draw(reference).rectangle(rectangle_box, fill=1)
    reference_mask = np.asarray(reference, dtype=np.bool_)
    image_name = 'synthetic_scene_320x240.png'
    sample_kind = 'synthetic'

points, point_labels = [[POINT_X, POINT_Y]], [1]
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'points': points, 'point_labels': point_labels, 'multimask': MULTIMASK, 'reference_mask_area_px': None if reference_mask is None else int(reference_mask.sum())})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `segment` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, at least one of points or box, 1..`MAX_PROMPTS` clicks that lie inside the image with 0/1 labels, one non-empty xyxy box inside the image, and a boolean `multimask` — and returns an **input manifest** naming the schema and ceilings, the input's observed mode and size, how many clicks and whether a box was given, the cleaned prompt values, and the verdict. The manifest is written to `outputs/sam_vit_segmentation_input_manifest.json`. To show what rejection looks like, the cell also validates a click outside the image and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB, resized and padded by the processor; masks are mapped back to input pixels with the padding stripped, and nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PROMPTS': MAX_PROMPTS, 'NUM_MULTIMASK_OUTPUTS': NUM_MULTIMASK_OUTPUTS, 'MASK_THRESHOLD': MASK_THRESHOLD}})
input_manifest = validate_inputs(image, points=points, point_labels=point_labels, multimask=MULTIMASK, names=[image_name])
# Demonstrate rejection on a prompt outside the image; the finding is recorded, not swallowed.
try:
    validate_inputs(image, points=[[image.width, image.height]], point_labels=[1])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'click-outside-image-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/sam_vit_segmentation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Segment one object and read the scores correctly

`segment` returns a dict with `masks` — a boolean array of shape `(K, H, W)` at input resolution, `K = 3` with `multimask=True` else `1` — `iou_scores` (one per mask), the cleaned `points`, `point_labels` and `box`, `multimask`, `width`, `height` and the model identity. Each `iou_scores` entry is the **model's own prediction** of how well that candidate overlaps the intended object: a learned, **uncalibrated** estimate, not a measured IoU and not a probability, produced by an unclipped regression head — **a value may exceed 1.0**. The conventional decision rule — used below and recorded in the evaluation report — is to keep the candidate with the highest predicted IoU; the pipeline ships no threshold, does not choose for you, and the caller owns any acceptance rule for their deployment. Masks are binarised at logit 0 (the processor default). Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place and, on ambiguous clicks, change which candidate ranks first. As recorded in the model card, the repository's CPU smoke on this same scene and click returned `iou_scores` of `[0.955, 1.012, 0.979]` with mask areas `[17129, 12216, 11887]` px against the rectangle's 12,221 px, argmax candidate 1 and `mask_iou` 1.000 at three decimals — note that the best score is above 1.0; that is one observation, not a calibration point.

In [ ]:
result = pipe.segment(image, points=points, point_labels=point_labels, multimask=MULTIMASK)
masks = result['masks']
best = int(np.argmax(result['iou_scores']))
best_mask = masks[best]
print({'masks_shape': masks.shape, 'iou_scores_model_predicted': [round(v, 4) for v in result['iou_scores']], 'scores_above_one': [i for i, v in enumerate(result['iou_scores']) if v > 1.0], 'mask_areas_px': [int(m.sum()) for m in masks], 'best_candidate': best, 'device': pipe.device})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No segmentation metric is reported by default: mean IoU needs labelled masks, and this repository ships none. The repository's only metric helper is `mask_iou(a, b)` (intersection-over-union of two boolean masks), the primitive a caller would use to compute mIoU on their own labelled data; when a reference mask is supplied the report carries one `mask_iou` entry per candidate — marking which candidate the decision rule selected — with the verdict `sample-sanity`. On the synthetic path that reference is a rectangle **you drew yourself**, so a high IoU proves only that the prompt contract, forward pass and up-sampling round-trip on a trivially separable shape. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable: hand-labelled masks on your own images averaged into a mean IoU over a held-out set. The report also carries the model-predicted IoU scores and every candidate's area, and it is written to `outputs/sam_vit_segmentation_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, reference_mask, sample_kind=sample_kind)
with open('outputs/sam_vit_segmentation_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No reference mask exists for this input, so mask_iou is not computed; inspect the exported mask and overlay instead.')

## 8. Export outputs and provenance

The best candidate mask is written as a 1-bit PNG (`outputs/sam_vit_segmentation_mask.png`) — the actual artifact a downstream consumer wants — and an overlay PNG paints it over the input for visual inspection (a supplement to, not a replacement for, the machine-readable files). JSON preserves every candidate's model-predicted IoU and area, the chosen candidate, the prompt, the evaluation report, the input manifest, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The boolean mask array itself is not embedded in the JSON — the PNG and its SHA-256 carry it. No credentials are recorded.

In [ ]:
Image.fromarray(best_mask).save('outputs/sam_vit_segmentation_mask.png')
overlay = np.asarray(image.convert('RGB')).copy()
overlay[best_mask] = (0.5 * overlay[best_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
Image.fromarray(overlay, mode='RGB').save('outputs/sam_vit_segmentation_overlay.png')
payload = {
    'prediction': {key: value for key, value in result.items() if key != 'masks'},
    'candidates': [{'index': i, 'iou_score_model_predicted': float(result['iou_scores'][i]), 'area_px': int(masks[i].sum())} for i in range(masks.shape[0])],
    'best_candidate': best,
    'mask_file': 'outputs/sam_vit_segmentation_mask.png',
    'mask_sha256': hashlib.sha256(np.packbits(best_mask).tobytes()).hexdigest(),
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'reference_mask_area_px': None if reference_mask is None else int(reference_mask.sum())},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/sam_vit_segmentation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The masks are the model's answer to *your* prompt: a click that is ambiguous returns candidates at several granularities, and the `iou_scores` used to rank them are the model's own uncalibrated estimates — unclipped, so occasionally above 1.0 — not measured overlaps. On the synthetic scene the `mask_iou` values in the evaluation report compare candidates with a rectangle you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, prompt validation, forward pass and up-sampling work on a trivially separable shape; it says nothing about photographs, thin structures, transparent or occluded objects, or clicks near a boundary, and a BYOD result is a single-image observation with the verdict `not-measurable`. One object per call, prompted mode only, no text prompts, no class labels. The pipeline provides no automatic mask generation, mask-input prompts, mIoU evaluation, or training capability.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/sam-vit-base/` and rerun Section 3. A `ValueError` naming the image sides or the click in Section 5: resize the BYOD image into 16–4096 px per side, or move `POINT_X`/`POINT_Y` inside it, and rerun from Section 4.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** set `MULTIMASK = False` and compare the single mask with the best candidate; prompt the disc with a box instead of a click (`pipe.segment(image, box=[200, 80, 281, 161], multimask=False)`) and compare its area with the disc's (the smoke run returned 5,132 px against the drawn 5,145 px, `mask_iou` 0.997, `iou_scores` `[1.002]` — another score above 1.0); add a background click (label 0) inside the rectangle after a foreground click on the disc to see the mask exclude it; enable `USE_BYOD` with a photograph, hand-draw one reference mask and pass it to `evaluation_report` to see the verdict switch to `sample-sanity` — the first step towards a real mIoU. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/facebook/sam-vit-base
- Upstream code: https://github.com/facebookresearch/segment-anything
- Segment Anything (Kirillov et al., ICCV 2023): https://arxiv.org/abs/2304.02643